# 5Essentials Data Collection and Processing

**Author:** An Nisa Astuti (annisaastuti@uchicago.edu)

**Team:** An Nisa Astuti and Yi Wang

**Description:** Web scraping pipeline for Chicago Public Schools 5Essentials survey data. Extracts school climate measures (2022–2025) for ~600 schools and aggregates to community areas for spatial analysis.

**AI statement:** I used ChatGPT  to explore Selenium-based approaches because my first BeautifulSoup attempts returned empty results. After discovering that the data was embedded as static JSON within HTML script tags, I pivoted back to a simpler requests + BeautifulSoup approach and wrote the extraction logic myself. 

**Key discovery**: The 5Essentials website embeds all school data as JSON in a `<script id="app_data">` tag.

**Data structure**:
- `essentials`: List of 5 Essential categories with scores
- `measures`: List of 22+ measures with detailed scores
- `target`: School metadata (name, address, response rates)

## Setup and Imports

In [2]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
import json
import html
import time
from pathlib import Path


## Configuration

In [3]:
CONFIG = {
    "base_url": "https://www.5-essentials.org/cps/5e/2025",
    "user_agent": "small scraper for teaching purposes annisaastutin@uchicago.edu",
    "request_timeout": 10,
    "delay_between_requests": 1.0,
}

HEADERS = {"User-Agent": CONFIG["user_agent"]}

# Rating code to label mapping (from the embedded data)
RATING_LABELS = {
    "X": "not eligible",
    "0": "no report",
    "1": "very weak",
    "2": "weak",
    "3": "neutral",
    "4": "strong",
    "5": "very strong",
    "P": "partial data"
}

## Step 1: Extract School IDs from Main Listing

In [4]:
def load_html_file(filepath):
    """
    Load and parse a local HTML file.
    
    Args:
        filepath: Path to the HTML file
        
    Returns:
        BeautifulSoup: Parsed HTML document
    """
    with open(filepath, encoding="utf-8") as f:
        return BeautifulSoup(f, "html.parser")

In [5]:
def extract_school_id(row):
    """
    Extract school ID from a table row.
    
    Args:
        row: BeautifulSoup <tr> element
        
    Returns:
        str: School ID, or None if not found
    """
    link = row.find("a")
    if link and link.get("href"):
        # URL format: /cps/5e/2025/s/{school_id}/...
        parts = link.get("href").strip("/").split("/")
        try:
            s_index = parts.index("s")
            return parts[s_index + 1]
        except (ValueError, IndexError):
            return None
    return None

In [6]:
def extract_school_ids_from_table(soup):
    """
    Extract all school IDs from the main listing table.
    
    Args:
        soup: BeautifulSoup object of the main listing page
        
    Returns:
        list[str]: List of school IDs
    """
    table = soup.find("table", class_="table no-select")
    if not table:
        raise ValueError("Could not find school listing table")
    
    rows = table.find_all("tr")[1:]  # Skip header
    school_ids = [extract_school_id(row) for row in rows]
    return [sid for sid in school_ids if sid]  # Filter None values

In [7]:
# Sample usage to extract first 5 school
HTML_PATH = "/Users/annisa/Library/CloudStorage/OneDrive-TheUniversityofChicago/Coursework/2026_Winter/30112_Python_Data_Management/30112-final-project/html-files-school/5Essentials Survey Home.html"

soup = load_html_file(HTML_PATH)
school_ids = extract_school_ids_from_table(soup)
print(f"Found {len(school_ids)} schools")
print(f"First 5: {school_ids[:5]}")

Found 648 schools
First 5: ['610229', '400172', '400013', '400017', '610038']


## Step 2: Build School URLs

In [8]:
def build_school_url(school_id, base_url=CONFIG["base_url"]):
    """
    Build URL for a school's essentials page.
    
    Args:
        school_id: School's unique identifier
        base_url: Base URL for 5Essentials site
        
    Returns:
        str: Full URL to school's essentials page
    """
    return f"{base_url}/s/{school_id}/essentials/"

In [9]:
# Build URLs using list comprehension
school_urls = [build_school_url(sid) for sid in school_ids]
print(f"First 3 URLs: {school_urls[:3]}")

First 3 URLs: ['https://www.5-essentials.org/cps/5e/2025/s/610229/essentials/', 'https://www.5-essentials.org/cps/5e/2025/s/400172/essentials/', 'https://www.5-essentials.org/cps/5e/2025/s/400013/essentials/']


## Step 3: Fetch and Extract JSON Data

The key insight: all data is embedded in `<script id="app_data" data-data="{...}">` as JSON.

In [10]:
def fetch_page(url, headers=HEADERS, timeout=CONFIG["request_timeout"]):
    """
    Fetch a web page and return BeautifulSoup object.
    
    Args:
        url: URL to fetch
        headers: Request headers
        timeout: Request timeout in seconds
        
    Returns:
        BeautifulSoup: Parsed HTML, or None if failed
    """
    try:
        response = requests.get(url, headers=headers, timeout=timeout)
        response.raise_for_status()
        return BeautifulSoup(response.text, "html.parser")
    except requests.RequestException as e:
        print(f"Error fetching {url}: {e}")
        return None

In [11]:
def extract_json_data(soup):
    """
    Extract the embedded JSON data from a school's page.
    
    The data is stored in: <script id="app_data" data-data="{...}">
    The JSON is HTML-encoded (using &quot; for quotes).
    
    Args:
        soup: BeautifulSoup object of a school's essentials page
        
    Returns:
        dict: Parsed JSON data, or None if not found
    """
    script_tag = soup.find("script", id="app_data")
    if not script_tag:
        return None
    
    data_str = script_tag.get("data-data")
    if not data_str:
        return None
    
    # Decode HTML entities (&quot; -> ")
    decoded = html.unescape(data_str)
    
    try:
        return json.loads(decoded)
    except json.JSONDecodeError as e:
        print(f"JSON decode error: {e}")
        return None

In [12]:
def fetch_school_data(url):
    """
    Fetch a school page and extract its JSON data.
    
    Combines fetch_page() and extract_json_data() into one step.
    
    Args:
        url: URL of the school's essentials page
        
    Returns:
        dict: Parsed JSON data, or None if failed
    """
    soup = fetch_page(url)
    if soup:
        return extract_json_data(soup)
    return None

## Step 4: Parse Essentials and Measures from JSON

The JSON structure contains:
- `essentials`: List of 5 Essential dicts with `slug`, `name`, `data`
- `measures`: List of 22+ measure dicts with `slug`, `name`, `data`, `parent_slug`
- `target`: School info (name, address, response rates)

In [13]:
def score_to_rating(score):
    """
    Convert a numeric score to a rating label.
    
    Score ranges (from the embedded config):
    - 0-20: very weak (1)
    - 20-40: weak (2)  
    - 40-60: neutral (3)
    - 60-80: strong (4)
    - 80-100: very strong (5)
    - Negative values: special codes (no data, partial, etc.)
    
    Args:
        score: Numeric score from the data
        
    Returns:
        str: Rating label
    """
    if score is None:
        return "no data"
    if score < 0:  # Special codes
        if score == -1000:
            return "no report"
        elif score == -1100:
            return "no report"
        elif score == -1200:
            return "not eligible"
        elif score == -1300:
            return "partial data"
        else:
            return "no report"
    if score < 20:
        return "very weak"
    if score < 40:
        return "weak"
    if score < 60:
        return "neutral"
    if score < 80:
        return "strong"
    return "very strong"

In [14]:
def parse_school_metadata(json_data):
    """
    Extract school metadata from JSON.
    
    Args:
        json_data: Parsed JSON from extract_json_data()
        
    Returns:
        dict: School metadata (id, name, address, response rates)
    """
    target = json_data.get("target", {})
    return {
        "school_id": target.get("slug"),
        "school_name": target.get("name"),
        "short_name": target.get("short_name"),
        "school_type": target.get("type_description"),
        "address": target.get("full_address"),
        "student_response_rate": target.get("student_response_rate"),
        "teacher_response_rate": target.get("teacher_response_rate"),
        "parent_response_rate": target.get("parent_response_rate"),
    }

In [15]:
def parse_essentials(json_data):
    """
    Extract the 5 Essential scores from JSON for all years.
    
    Args:
        json_data: Parsed JSON data
        
    Returns:
        dict: Essential slug -> {name, score_2025, rating_2025, ...}
    """
    essentials = {}
    
    for ess in json_data.get("essentials", []):
        slug = ess.get("slug")
        if slug in ["overall", "_supplemental"]:
            continue
            
        data = ess.get("data", {}).get("all", [])
        
        score_2025 = data[0][0] if len(data) > 0 else None
        score_2024 = data[1][0] if len(data) > 1 else None
        score_2023 = data[2][0] if len(data) > 2 else None
        score_2022 = data[3][0] if len(data) > 3 else None
        
        essentials[slug] = {
            "name": ess.get("name"),
            # 2025
            "score_2025": score_2025,
            "rating_2025": score_to_rating(score_2025),
            # 2024
            "score_2024": score_2024,
            "rating_2024": score_to_rating(score_2024),
            # 2023
            "score_2023": score_2023,
            "rating_2023": score_to_rating(score_2023),
            # 2022
            "score_2022": score_2022,
            "rating_2022": score_to_rating(score_2022),
        }
    
    return essentials

In [16]:
def parse_measures(json_data):
    """
    Extract all measure scores from JSON.
    
    Each measure has:
    - slug: identifier (e.g., 'inf24' for Teacher Influence)
    - name: display name
    - parent_slug: which Essential it belongs to
    - data.all: scores for each year
    - data contains subgroup breakdowns (boys, girls, grade4, etc.)
    
    Args:
        json_data: Parsed JSON data
        
    Returns:
        list[dict]: List of measure dicts with scores
    """
    measures = []
    
    for meas in json_data.get("measures", []):
        slug = meas.get("slug")
        if slug == "all":  # Skip the 'all measures' placeholder
            continue
            
        data = meas.get("data", {})
        all_scores = data.get("all", [])
        
        score_2025 = all_scores[0][0] if len(all_scores) > 0 else None
        score_2024 = all_scores[1][0] if len(all_scores) > 1 else None
        score_2023 = all_scores[2][0] if len(all_scores) > 2 else None
        score_2022 = all_scores[3][0] if len(all_scores) > 3 else None
        
        measure = {
            "measure_slug": slug,
            "measure_name": meas.get("name"),
            "parent_essential": meas.get("parent_slug"),
            "subject": meas.get("subject"),  # 1=Teacher, 2=Student
            # 2025
            "score_2025": score_2025,
            "rating_2025": score_to_rating(score_2025),
            # 2024
            "score_2024": score_2024,
            "rating_2024": score_to_rating(score_2024),
            # 2023
            "score_2023": score_2023,
            "rating_2023": score_to_rating(score_2023),
            # 2022
            "score_2022": score_2022,
            "rating_2022": score_to_rating(score_2022),
        }
        
        measures.append(measure)
    
    return measures

## Step 5: Combine into Full School Data

In [17]:
def parse_school_json(json_data):
    """
    Parse all components from a school's JSON data.
    
    Args:
        json_data: Parsed JSON from extract_json_data()
        
    Returns:
        dict: Complete school data with metadata, essentials, measures
    """
    if not json_data:
        return None
        
    return {
        "metadata": parse_school_metadata(json_data),
        "essentials": parse_essentials(json_data),
        "measures": parse_measures(json_data),
    }

In [18]:
def scrape_school(url):
    """
    Scrape and parse a single school's data.
    
    Combines fetching and parsing into one function.
    
    Args:
        url: URL to the school's essentials page
        
    Returns:
        dict: Parsed school data, or None if failed
    """
    json_data = fetch_school_data(url)
    return parse_school_json(json_data)

## Step 6: Scrape All Schools



In [19]:
def scrape_school_with_delay(url, delay=CONFIG["delay_between_requests"]):
    """
    Scrape a school and wait before returning.
    
    Args:
        url: School URL
        delay: Seconds to wait after scraping
        
    Returns:
        dict: Parsed school data
    """
    result = scrape_school(url)
    time.sleep(delay)
    return result

In [20]:
all_data = []
total = len(school_urls)

for i, url in enumerate(school_urls):
    if (i + 1) % 50 == 0:
        print(f"Progress: {i + 1}/{total}")
    all_data.append(scrape_school_with_delay(url))

print("Done")

Progress: 50/648
Progress: 100/648
Progress: 150/648
Progress: 200/648
Progress: 250/648
Progress: 300/648
Progress: 350/648
Progress: 400/648
Progress: 450/648
Progress: 500/648
Progress: 550/648
Progress: 600/648
Done


In [21]:
# Check
success = 0
failed = 0

for data in all_data:
    if data:
        success += 1
    else:
        failed += 1

print(f"Successfully scraped: {success} schools")
print(f"Failed: {failed} schools")
print(f"Total measures per school: {len(all_data[0]['measures']) if all_data[0] else 'N/A'}")

Successfully scraped: 648 schools
Failed: 0 schools
Total measures per school: 37


## Step 7: Flatten to DataFrames

In [22]:
def flatten_to_wide(school_data_list):
    """
    Flatten school data to wide format (one row per school).
    
    Columns: school_id, school_name, address, essential_leaders_score_2025, ...
    
    Args:
        school_data_list: List of parsed school dicts
        
    Returns:
        pd.DataFrame: Wide format DataFrame
    """
    rows = []
    
    for school in school_data_list:
        if not school:
            continue
            
        row = school["metadata"].copy()
        
        # Add essential scores for all years
        for slug, ess_data in school["essentials"].items():
            # 2025
            row[f"essential_{slug}_score_2025"] = ess_data.get("score_2025")
            row[f"essential_{slug}_rating_2025"] = ess_data.get("rating_2025")
            # 2024
            row[f"essential_{slug}_score_2024"] = ess_data.get("score_2024")
            row[f"essential_{slug}_rating_2024"] = ess_data.get("rating_2024")
            # 2023
            row[f"essential_{slug}_score_2023"] = ess_data.get("score_2023")
            row[f"essential_{slug}_rating_2023"] = ess_data.get("rating_2023")
            # 2022
            row[f"essential_{slug}_score_2022"] = ess_data.get("score_2022")
            row[f"essential_{slug}_rating_2022"] = ess_data.get("rating_2022")
        
        rows.append(row)
    
    return pd.DataFrame(rows)

In [23]:
def flatten_measures_long(school_data_list):
    """
    Flatten to long format (one row per school-measure).
    
    Useful for analysis comparing measures across schools.
    
    Args:
        school_data_list: List of parsed school dicts
        
    Returns:
        pd.DataFrame: Long format DataFrame
    """
    rows = []
    
    for school in school_data_list:
        if not school:
            continue
            
        meta = school["metadata"]
        
        for measure in school["measures"]:
            row = {
                "school_id": meta["school_id"],
                "school_name": meta["school_name"],
                "school_type": meta["school_type"],
                **measure  # Spread all measure fields
            }
            rows.append(row)
    
    return pd.DataFrame(rows)

In [24]:
# Create DataFrames from sample
df_wide = flatten_to_wide(all_data)
df_long = flatten_measures_long(all_data)

print(f"Wide: {df_wide.shape}")
print(f"Long: {df_long.shape}")

Wide: (648, 48)
Long: (24312, 15)


In [25]:
# Preview wide format
df_wide.head()

,school_id,school_name,short_name,school_type,address,student_response_rate,teacher_response_rate,parent_response_rate,essential_leaders_score_2025,essential_leaders_rating_2025,...,essential_environment_score_2022,essential_environment_rating_2022,essential_instruction_score_2025,essential_instruction_rating_2025,essential_instruction_score_2024,essential_instruction_rating_2024,essential_instruction_score_2023,essential_instruction_rating_2023,essential_instruction_score_2022,essential_instruction_rating_2022
0,610229,A.N. Pritzker School,Pritzker,Elementary School (K/PK-8),"2009 W Schiller St, Chicago, IL 60622",80.1,69.1,0.0,60.0,strong,...,48.0,neutral,-1300.0,partial data,50.0,neutral,49.0,neutral,43.0,neutral
1,400172,ASPIRA Business and Finance,ASPIRA - Business and Finance,High School (9-12),"2989 N Milwaukee Ave, Chicago, IL 60618",88.1,89.6,0.0,43.0,neutral,...,48.0,neutral,-1300.0,partial data,54.0,neutral,47.0,neutral,43.0,neutral
2,400013,ASPIRA Charter School - Early College High School,ASPIRA Charter - Early College,High School (9-12),"3986 W Barry Ave, Chicago, IL 60618",98.9,84.6,0.0,69.0,strong,...,58.0,neutral,-1300.0,partial data,47.0,neutral,53.0,neutral,52.0,neutral
3,400017,ASPIRA Charter School - Haugan Middle School,ASPIRA Charter - Haugan,Middle School (6-8),"3729 W Leland Ave, Chicago, IL 60625",99.9,88.9,0.0,47.0,neutral,...,57.0,neutral,-1300.0,partial data,58.0,neutral,58.0,neutral,48.0,neutral
4,610038,Abraham Lincoln Elementary School,Lincoln,Elementary School (K/PK-8),"615 W Kemper Pl, Chicago, IL 60614",92.3,58.7,0.0,40.0,neutral,...,63.0,strong,-1300.0,partial data,60.0,strong,66.0,strong,59.0,neutral


In [26]:
# Preview long format
df_long.head(10)

,school_id,school_name,school_type,measure_slug,measure_name,parent_essential,subject,score_2025,rating_2025,score_2024,rating_2024,score_2023,rating_2023,score_2022,rating_2022
0,610229,A.N. Pritzker School,Elementary School (K/PK-8),engg24,Academic Engagement,_supplemental,2,-1000.0,no report,37.0,weak,44.0,neutral,47.0,neutral
1,610229,A.N. Pritzker School,Elementary School (K/PK-8),perc24,Academic Personalism,environment,2,-1000.0,no report,35.0,weak,40.0,neutral,40.0,neutral
2,610229,A.N. Pritzker School,Elementary School (K/PK-8),pres24,Academic Press,instruction,2,-1000.0,no report,50.0,neutral,56.0,neutral,40.0,neutral
3,610229,A.N. Pritzker School,Elementary School (K/PK-8),cdis24,Classroom Disruptions,_supplemental,1,55.0,neutral,64.0,strong,48.0,neutral,55.0,neutral
4,610229,A.N. Pritzker School,Elementary School (K/PK-8),rigr24,Classroom Rigor,_supplemental,2,-1000.0,no report,48.0,neutral,56.0,neutral,45.0,neutral
5,610229,A.N. Pritzker School,Elementary School (K/PK-8),colb24,Collaborative Practices,teachers,1,66.0,strong,48.0,neutral,32.0,weak,40.0,neutral
6,610229,A.N. Pritzker School,Elementary School (K/PK-8),colr24,Collective Responsibility,teachers,1,52.0,neutral,49.0,neutral,38.0,weak,44.0,neutral
7,610229,A.N. Pritzker School,Elementary School (K/PK-8),clar24,Course Clarity,_supplemental,2,-1000.0,no report,51.0,neutral,50.0,neutral,47.0,neutral
8,610229,A.N. Pritzker School,Elementary School (K/PK-8),emhl24,Emotional Health,_supplemental,2,48.0,neutral,51.0,neutral,48.0,neutral,48.0,neutral
9,610229,A.N. Pritzker School,Elementary School (K/PK-8),engl24,English Instruction,instruction,2,44.0,neutral,47.0,neutral,56.0,neutral,38.0,weak


In [27]:
# Save to CSV
df_wide.to_csv("5essentials_schools_wide.csv", index=False)
df_long.to_csv("5essentials_measures_long.csv", index=False)

## Full Pipeline

## Notes

### Data Available
The embedded JSON contents:
- **Subgroup breakdowns**: scores by gender, grade, race, IEP status, ELL status, FRL status
- **Historical data**: 4 years of scores (2022-2025)
- **Comparison data**: district and similar school averages

### For Community Area Mapping
The `address` field in metadata can be geocoded to latitude/longitude, then mapped to Chicago community areas using the city's community area boundaries shapefile.

### Subject Codes
- `1` = Teacher survey measure
- `2` = Student survey measure  
- `6` = Parent survey measure

### Special Score Values
- `-1000`: No report (insufficient responses)
- `-1100`: No report
- `-1200`: Not eligible
- `-1300`: Partial data (some questions missing)